In [1]:
# =============================================
# 03_Model_Training.ipynb
# =============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data
df_clean = pd.read_csv('data/cleaned_student_data.csv')
df = pd.read_csv('data/processed_student_data.csv')  # for target

print("✅ Data Loaded!")
print("Features:", df_clean.shape[1])

ModuleNotFoundError: No module named 'xgboost'

In [3]:
# Install XGBoost (Run this cell first)
!pip install xgboost

print("✅ XGBoost installed successfully!")

✅ XGBoost installed successfully!


In [1]:
# =============================================
# 03_Model_Training.ipynb
# =============================================

!pip install xgboost -q   # Install if needed

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df_clean = pd.read_csv('data/cleaned_student_data.csv')
df = pd.read_csv('data/processed_student_data.csv')   # for target

print("✅ Libraries and Data Loaded!")
print("Number of features:", df_clean.shape[1])

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


✅ Libraries and Data Loaded!
Number of features: 45


In [2]:
# Install XGBoost (Run this cell)
!pip install xgboost -q

print("✅ XGBoost is ready!")

✅ XGBoost is ready!


In [3]:
# =============================================
# TRAIN XGBoost MODEL
# =============================================

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib

# Prepare data
X = df_clean.copy()
y = df['at_risk']

print("Features:", X.shape[1])
print("Target distribution:\n", y.value_counts())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc'
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("🚀 MODEL PERFORMANCE")
print("="*60)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}   ← Most Important")
print(f"AUC       : {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

joblib.dump(model, 'xgboost_atrisk_model.pkl')
print("\n✅ Model saved!")

Features: 45
Target distribution:
 at_risk
1    262
0    133
Name: count, dtype: int64

🚀 MODEL PERFORMANCE
Accuracy  : 0.8687
Recall    : 0.9242   ← Most Important
AUC       : 0.9229

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.76      0.79        33
           1       0.88      0.92      0.90        66

    accuracy                           0.87        99
   macro avg       0.86      0.84      0.85        99
weighted avg       0.87      0.87      0.87        99


✅ Model saved!


In [4]:
# =============================================
# IMPROVED XGBoost MODEL (Hyperparameter Tuning + Class Weight)
# =============================================

from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, classification_report
import joblib

X = df_clean.copy()
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Improved Model with Class Weight (helps with imbalance)
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])   # Handle imbalance
)

# Optional: Quick Hyperparameter tuning (takes 30-60 seconds)
param_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [300, 500]
}

grid_search = GridSearchCV(model, param_grid, cv=3, scoring='recall', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# Evaluate
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("="*70)
print("🚀 IMPROVED MODEL PERFORMANCE")
print("="*70)
print(f"Best Params : {grid_search.best_params_}")
print(f"Accuracy    : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall      : {recall_score(y_test, y_pred):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save best model
joblib.dump(best_model, 'xgboost_atrisk_improved.pkl')
print("\n✅ Improved Model Saved!")

🚀 IMPROVED MODEL PERFORMANCE
Best Params : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
Accuracy    : 0.8788
Recall      : 0.9091
AUC-ROC     : 0.9247

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82        33
           1       0.91      0.91      0.91        66

    accuracy                           0.88        99
   macro avg       0.86      0.86      0.86        99
weighted avg       0.88      0.88      0.88        99


✅ Improved Model Saved!


In [5]:
# =============================================
# ADVANCED MODEL IMPROVEMENT
# =============================================

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import joblib
import numpy as np

X = df_clean.copy()
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y   # Smaller test size
)

# Advanced Hyperparameter Search
param_dist = {
    'n_estimators': [400, 600, 800],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.08],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2]
}

xgb = XGBClassifier(
    random_state=42,
    eval_metric='auc',
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
)

random_search = RandomizedSearchCV(
    xgb, param_distributions=param_dist, 
    n_iter=30, cv=4, scoring='f1', 
    n_jobs=-1, random_state=42
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_

# Final Evaluation
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("="*70)
print("🚀 ADVANCED IMPROVED MODEL")
print("="*70)
print("Best Parameters:", random_search.best_params_)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

joblib.dump(best_model, 'xgboost_atrisk_best.pkl')
print("\n✅ Best Model Saved!")

🚀 ADVANCED IMPROVED MODEL
Best Parameters: {'subsample': 0.8, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.7}
Accuracy  : 0.8228
Recall    : 0.8462
AUC       : 0.9046

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.78      0.75        27
           1       0.88      0.85      0.86        52

    accuracy                           0.82        79
   macro avg       0.80      0.81      0.81        79
weighted avg       0.83      0.82      0.82        79


✅ Best Model Saved!


In [6]:
# =============================================
# BEST STABLE MODEL (Recommended)
# =============================================

from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import joblib

X = df_clean.copy()
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.8,
    gamma=0.1,
    random_state=42,
    eval_metric='auc',
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("="*60)
print("✅ BEST STABLE MODEL PERFORMANCE")
print("="*60)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 5-Fold Cross Validation Score
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"\n5-Fold CV Accuracy: {cv_scores.mean():.4f} (± {cv_scores.std():.4f})")

joblib.dump(model, 'xgboost_atrisk_final.pkl')
print("\n✅ Final Model Saved!")

✅ BEST STABLE MODEL PERFORMANCE
Accuracy  : 0.8228
Recall    : 0.8462
AUC       : 0.9060

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.78      0.75        27
           1       0.88      0.85      0.86        52

    accuracy                           0.82        79
   macro avg       0.80      0.81      0.81        79
weighted avg       0.83      0.82      0.82        79


5-Fold CV Accuracy: 0.8911 (± 0.0129)

✅ Final Model Saved!


In [7]:
# =============================================
# FINAL MODEL + FEATURE SELECTION
# =============================================

from sklearn.feature_selection import SelectFromModel

# Use the best model from before
X = df_clean.copy()
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.22, random_state=42, stratify=y
)

# Train model with feature selection
final_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.07,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.75,
    gamma=0.1,
    random_state=42,
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
)

# Select important features
selector = SelectFromModel(final_model, prefit=False, threshold='median')
selector.fit(X_train, y_train)

X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

print(f"Features reduced from {X.shape[1]} → {X_train_selected.shape[1]}")

# Train on selected features
final_model.fit(X_train_selected, y_train)

y_pred = final_model.predict(X_test_selected)
y_pred_proba = final_model.predict_proba(X_test_selected)[:, 1]

print("\n" + "="*65)
print("🏆 FINAL OPTIMIZED MODEL")
print("="*65)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

joblib.dump(final_model, 'xgboost_atrisk_final.pkl')
joblib.dump(selector, 'feature_selector.pkl')
print("\n✅ Final Model + Selector Saved!")

Features reduced from 45 → 23

🏆 FINAL OPTIMIZED MODEL
Accuracy  : 0.8621
Recall    : 0.8966
AUC       : 0.8918

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.79      0.79        29
           1       0.90      0.90      0.90        58

    accuracy                           0.86        87
   macro avg       0.84      0.84      0.84        87
weighted avg       0.86      0.86      0.86        87


✅ Final Model + Selector Saved!


In [8]:
# Load and save the best performing model
best_model = joblib.load('xgboost_atrisk_improved.pkl')   # or the one with 87.88%

joblib.dump(best_model, 'xgboost_atrisk_final_best.pkl')
print("✅ Best Model (87.88%) saved as final!")

✅ Best Model (87.88%) saved as final!
